# Traditional Chatbot with AI Sentiment Integration


In [1]:
!pip -q install flask flask-cors transformers torch pyngrok requests

## 1. Loading the local sentiment model

In [2]:
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=-1
)

def analyze_sentiment(message):
    result = sentiment_model(message, truncation=True, max_length=512)[0]
    return {"sentiment": result["label"], "confidence": round(float(result["score"]), 4)}

for message in ["I am very happy today!", "I am disappointed with this service.", "This chatbot is amazing!"]:
    print(message, analyze_sentiment(message))

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

I am very happy today! {'sentiment': 'POSITIVE', 'confidence': 0.9999}
I am disappointed with this service. {'sentiment': 'NEGATIVE', 'confidence': 0.9997}
This chatbot is amazing! {'sentiment': 'POSITIVE', 'confidence': 0.9999}


## 2. Starting Flask with CORS


In [3]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from werkzeug.serving import make_server
import threading

app = Flask(__name__)

# Allow the published Hugging Face static Space to call this API.
ALLOWED_ORIGINS = [
    "https://haripriya2030-traditional-ai-chatbot-gradio.static.hf.space",
    "https://huggingface.co",
]
CORS(app, resources={r"/analyze": {
    "origins": ALLOWED_ORIGINS,
    "methods": ["POST", "OPTIONS"],
    "allow_headers": ["Content-Type", "ngrok-skip-browser-warning"],
}})

@app.route("/analyze", methods=["POST"])
def analyze():
    data = request.get_json(silent=True) or {}
    message = data.get("message", "")
    if not isinstance(message, str) or not message.strip():
        return jsonify({"error": "Message is required"}), 400
    try:
        return jsonify(analyze_sentiment(message))
    except Exception as exc:
        app.logger.exception("Sentiment analysis failed")
        return jsonify({"error": "Analysis failed"}), 500

server = make_server("0.0.0.0", 5000, app, threaded=True)
server_thread = threading.Thread(target=server.serve_forever, daemon=True)
server_thread.start()
print("Flask API running on port 5000")

Flask API running on port 5000


## 3. Verifying local API and CORS preflight

In [4]:
import requests

local_url = "http://127.0.0.1:5000/analyze"
r = requests.post(local_url, json={"message": "I am very happy today!"}, timeout=30)
print("Local POST:", r.status_code, r.json())
assert r.status_code == 200

origin = "https://haripriya2030-traditional-ai-chatbot-gradio.static.hf.space"
preflight = requests.options(local_url, headers={
    "Origin": origin,
    "Access-Control-Request-Method": "POST",
    "Access-Control-Request-Headers": "content-type,ngrok-skip-browser-warning"
}, timeout=15)
print("Local OPTIONS:", preflight.status_code)
print("Allowed origin:", preflight.headers.get("Access-Control-Allow-Origin"))
print("Allowed headers:", preflight.headers.get("Access-Control-Allow-Headers"))
assert preflight.headers.get("Access-Control-Allow-Origin") == origin

INFO:werkzeug:127.0.0.1 - - [08/Nov/2026 09:10:41] "POST /analyze HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [08/Nov/2026 09:10:41] "OPTIONS /analyze HTTP/1.1" 200 -


Local POST: 200 {'confidence': 0.9999, 'sentiment': 'POSITIVE'}
Local OPTIONS: 200
Allowed origin: https://haripriya2030-traditional-ai-chatbot-gradio.static.hf.space
Allowed headers: content-type, ngrok-skip-browser-warning


## 4. Open ngrok tunnel


In [11]:

from google.colab import userdata
from pyngrok import ngrok, conf

ngrok_token = userdata.get("NGROK_AUTHTOKEN").strip()

# Use the token directly without rewriting ngrok.yml
config = conf.PyngrokConfig(
    auth_token=ngrok_token
)

# Stop any tunnels managed by this runtime
ngrok.kill()

# Start a new HTTPS tunnel
tunnel = ngrok.connect(
    addr=5000,
    proto="http",
    pyngrok_config=config
)

PUBLIC_API_URL = tunnel.public_url

print("Public AI API URL:", PUBLIC_API_URL)
print("ngrok tunnel started successfully!")


Public AI API URL: https://unsubtly-singular-dispute.ngrok-free.dev
ngrok tunnel started successfully!


## 5. Testing public POST and browser-style OPTIONS


In [12]:
public_endpoint = PUBLIC_API_URL.rstrip("/") + "/analyze"
headers = {"ngrok-skip-browser-warning": "true", "Origin": origin}
post = requests.post(public_endpoint, json={"message": "I am excited about learning AI!"}, headers=headers, timeout=45)
print("Public POST:", post.status_code, post.text[:400])
print("POST allowed origin:", post.headers.get("Access-Control-Allow-Origin"))

options = requests.options(public_endpoint, headers={
    "Origin": origin,
    "Access-Control-Request-Method": "POST",
    "Access-Control-Request-Headers": "content-type,ngrok-skip-browser-warning",
    "ngrok-skip-browser-warning": "true"
}, timeout=30)
print("Public OPTIONS:", options.status_code)
print("OPTIONS allowed origin:", options.headers.get("Access-Control-Allow-Origin"))
print("OPTIONS allowed headers:", options.headers.get("Access-Control-Allow-Headers"))
print("OPTIONS response preview:", options.text[:200])
if options.headers.get("Access-Control-Allow-Origin") != origin:
    print("WARNING: Browser preflight is not passing; inspect ngrok or origin configuration.")

INFO:werkzeug:127.0.0.1 - - [08/Nov/2026 09:21:49] "POST /analyze HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [08/Nov/2026 09:21:50] "OPTIONS /analyze HTTP/1.1" 200 -


Public POST: 200 {"confidence":0.9998,"sentiment":"POSITIVE"}

POST allowed origin: https://haripriya2030-traditional-ai-chatbot-gradio.static.hf.space
Public OPTIONS: 200
OPTIONS allowed origin: https://haripriya2030-traditional-ai-chatbot-gradio.static.hf.space
OPTIONS allowed headers: content-type, ngrok-skip-browser-warning
OPTIONS response preview: 


In [13]:
print(PUBLIC_API_URL)

https://unsubtly-singular-dispute.ngrok-free.dev
